#### Get box info

In [4]:
def get_box(pdb_id, ligand_position):
    lig_name, lig_chain, lig_index = ligand_position.split(":")
    ligand_rows = []
    with open(f"complexes/{pdb_id}/complex.pdb", "r") as pdb_file:
        while line := pdb_file.readline():
            if len(line) < 80:
                continue
            if (line[0:4] == "ATOM" or line[0:6] == "HETATM") and line[21] == lig_chain and line[22:26].strip() == lig_index: #  and line[17:20].strip() == lig_name
                ligand_rows.append(line)
    x_coords = []
    y_coords = []
    z_coords = []
    if not len(ligand_rows):
        return (None,) * 6
    for row in ligand_rows:
        x_coords.append(float(row[30:38].strip()))
        y_coords.append(float(row[38:46].strip()))
        z_coords.append(float(row[46:54].strip()))
    x_min, y_min, z_min = min(x_coords), min(y_coords), min(z_coords)
    x_max, y_max, z_max = max(x_coords), max(y_coords), max(z_coords)
    x_center, y_center, z_center = (x_max + x_min) / 2, (y_max + y_min) / 2, (z_max + z_min) / 2
    x_size, y_size, z_size = x_max - x_min + 10, y_max - y_min + 10, z_max - z_min + 10
    return round(x_center, 1), round(y_center, 1), round(z_center, 1), round(x_size, 1), round(y_size, 1), round(z_size, 1)

In [5]:
box_data = tuple(zip(*df_kd[["PDB ID", "PDB ligand position"]].apply(lambda x: get_box(x[0], x[1]), axis=1)))
df_kd["center_x"], df_kd["center_y"], df_kd["center_z"], df_kd["size_x"], df_kd["size_y"], df_kd["size_z"] = box_data

In [6]:
df_kd

,PDB ID,PDB ligand position,SMILES,pKd,center_x,center_y,center_z,size_x,size_y,size_z
0,185L,IND:A:400,c1ccc2c(c1)cc[nH]2,3.539102,27.3,6.2,4.2,12.6,12.2,14.2
1,186L,N4B:A:400,CCCCc1ccccc1,4.853872,27.1,7.2,3.1,12.5,14.4,15.1
2,187L,PXY:A:400,Cc1ccc(cc1)C,3.374688,27.1,6.9,3.7,12.0,13.4,14.2
3,188L,OXE:A:400,Cc1ccccc1C,3.328827,27.2,6.9,3.9,12.4,14.0,13.1
4,1A50,FIP:A:270,c1cc2c(cc1F)c(c[nH]2)CCCOP(=O)(O)O,6.455932,49.8,26.5,12.0,15.4,17.9,16.3
...,...,...,...,...,...,...,...,...,...,...
2161,7BQU,EF2:A:501,c1ccc2c(c1)C(=O)N(C2=O)[C@H]3CCC(=O)NC3=O,5.397940,25.1,17.7,13.3,14.7,15.6,17.6
2162,7BQV,F4U:A:501,c1cc2c(cc1O)C(=O)N(C2=O)[C@H]3CCC(=O)NC3=O,5.642065,-28.7,-18.3,-8.9,14.6,15.1,19.1
2163,7C2I,SAM:A:301,C[S@@+](CC[C@@H](C(=O)[O-])N)C[C@@H]1[C@H]([C@...,5.387216,55.0,-65.5,-1.0,23.3,15.0,17.9
2164,7KDR,J1L:A:201,CC1(C(=NC2=C(N1)N=C(NC2=O)N)C(=O)NCCN3CC[C@H](...,7.327902,10.4,12.3,20.8,18.3,30.9,20.0


In [8]:
df_kd.to_csv("box_data.csv", index=False)